In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Load dataset
df = pd.read_csv('Cust_segmentation.csv')

# Select features for clustering (replace with the actual column names you want to use)
features = ['CustomerId', 'Address']  # example features, replace accordingly
X = df[features]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Number of clusters
k = 3

# K-means using Euclidean distance (default distance metric)
kmeans_euc = KMeans(n_clusters=k, random_state=42)
kmeans_euc.fit(X_scaled)
labels_euc = kmeans_euc.labels_

# Function to perform K-means with Manhattan distance
def kmeans_manhattan(X, n_clusters, max_iter=300):
    # Initialize cluster centers randomly
    rng = np.random.RandomState(42)
    centers = X[rng.choice(X.shape[0], n_clusters, replace=False)]
    
    for i in range(max_iter):
        # Assign labels based on Manhattan distance
        labels = np.argmin(np.sum(np.abs(X[:, None] - centers[None, :]), axis=2), axis=1)
        
        new_centers = np.array([X[labels == j].mean(axis=0) for j in range(n_clusters)])
        
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    
    return labels, centers

labels_man, centers_man = kmeans_manhattan(X_scaled, k)

# Add labels to dataframe for plotting
df['Cluster_Euclidean'] = labels_euc
df['Cluster_Manhattan'] = labels_man

# 2D Plot for first two features with Euclidean clusters
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_euc, cmap='viridis')
plt.title('K-means Clusters (Euclidean)')
plt.xlabel(features[0])
plt.ylabel(features[1])

# 2D Plot for first two features with Manhattan clusters
plt.subplot(1, 2, 2)
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_man, cmap='viridis')
plt.title('K-means Clusters (Manhattan)')
plt.xlabel(features[0])
plt.ylabel(features[1])
plt.show()

# 3D Plot for first three features with Euclidean clusters
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(121, projection='3d')
ax.scatter(X_scaled[:, 0], X_scaled[:, 1], X_scaled[:, 2], c=labels_euc, cmap='viridis')
ax.set_title('3D Clusters (Euclidean)')
ax.set_xlabel(features[0])
ax.set_ylabel(features[1])
ax.set_zlabel(features[2])

# 3D Plot for Manhattan clusters
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(X_scaled[:, 0], X_scaled[:, 1], X_scaled[:, 2], c=labels_man, cmap='viridis')
ax2.set_title('3D Clusters (Manhattan)')
ax2.set_xlabel(features[0])
ax2.set_ylabel(features[1])
ax2.set_zlabel(features[2])

plt.show()
